# Rétropropagation à la main

Deux poids suffisent pour comprendre la règle de la chaîne.

Nous exécutons ce notebook avec un noyau Python neuf, sur CPU.

Version vérifiée : Python 3.11.9, PyTorch 2.10.0 et TorchAudio 2.10.0.

Les cellules n’installent aucun paquet et ne téléchargent aucune donnée.

Les nombres du cours viennent de cette version ; une autre version peut modifier certains arrondis ou messages.

Dans Colab, choisissons un environnement sans accélérateur, puis Exécution → Tout exécuter.

Les trous des exercices restent dans du Markdown : ils ne cassent donc pas l’exécution complète.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#retropropagation-a-la-main).

Le programme suivant est copié à l’identique depuis le cours.

In [1]:
import os
import sys
import tempfile

# À lancer dans un noyau neuf : ce tutoriel reste sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA visible :", torch.cuda.is_available())
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-pytorch-")
os.chdir(_quiz_temporary_directory.name)
assert not torch.cuda.is_available(), 'Choisir un noyau CPU neuf pour ce notebook.'

Python : 3.11.9
PyTorch : 2.10.0+cpu
CUDA visible : False


## Nous lisons le programme

**1. Construire les deux couches**

Les deux poids sont des tenseurs suivis. Le calcul direct reproduit les valeurs du schéma.

**2. Remonter depuis la perte**

backward traverse les multiplications et le carré. Nous ne lui donnons pas nos formules de dérivées.

**3. Comparer les valeurs**

Les gradients des poids coïncident avec les produits du tableau.

In [2]:
import torch
x = torch.tensor(2.)
w1 = torch.tensor(3., requires_grad=True)
w2 = torch.tensor(-1., requires_grad=True)
h = w1 * x
h.retain_grad()
y = w2 * h
y.retain_grad()
perte = (y - 0) ** 2
perte.backward()
print("h, y, L :", h.item(), y.item(), perte.item())
print("dL/dy, dL/dh :", y.grad.item(), h.grad.item())
print("w1.grad, w2.grad :", w1.grad.item(), w2.grad.item())

h, y, L : 6.0 -6.0 36.0
dL/dy, dL/dh : -12.0 12.0
w1.grad, w2.grad : 24.0 -72.0


## Interpréter avant de modifier

Exécution CPU vérifiée ; les sorties sont conservées dans scripts/verify/pytorch-additions-results.json.

La passe directe fixe les valeurs de chaque opération.

La passe arrière combine leurs dérivées locales.

Les gradients calculés automatiquement doivent rester explicables sur un petit cas.

## Exercice 1 — Écrire les dérivées locales

Retrouvons les deux gradients sans appeler PyTorch. Les valeurs sont celles du chapitre.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
x, w1, w2 = 2., 3., -1.
h = w1 * x
y = w2 * h
dL_dy = ... * y
dL_dw2 = dL_dy * ...
dL_dh = dL_dy * ...
dL_dw1 = dL_dh * ...
```

**Piste :** Sur chaque arête, nous multiplions le gradient reçu par la dérivée locale de l’opération.

## Exercice 2 — Modifier une seule cible

Gardons x, w1 et w2. La cible devient −2. Complétons la perte et vérifions la nouvelle passe arrière.

Copions ce squelette dans une nouvelle cellule, puis remplaçons les `...`.

```python
w1 = torch.tensor(3., requires_grad=True)
w2 = torch.tensor(-1., requires_grad=True)
y = w2 * (w1 * 2)
perte = (y - (...)).square()
perte....()
print(w1.grad.item(), w2.grad.item())
```

**Piste :** Le nouveau gradient reçu par y vaut 2 × (y − cible). Les deux autres dérivées locales ne changent pas.

In [3]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()
print('Programme terminé ; fichiers temporaires supprimés.')

Programme terminé ; fichiers temporaires supprimés.


## Solutions — ouvrons-les après notre recherche

Les solutions utilisent leurs propres variables. Nous pouvons les copier dans de nouvelles cellules.

<details>
<summary>Afficher la solution 1 — Écrire les dérivées locales</summary>

<pre><code>x_exo, w1_exo, w2_exo = 2.0, 3.0, -1.0
h_exo = w1_exo * x_exo
y_exo = w2_exo * h_exo
dL_dy_exo = 2 * y_exo
dL_dw2_exo = dL_dy_exo * h_exo
dL_dh_exo = dL_dy_exo * w2_exo
dL_dw1_exo = dL_dh_exo * x_exo
assert (dL_dw1_exo, dL_dw2_exo) == (24.0, -72.0)
print(&#x27;gradients manuels :&#x27;, dL_dw1_exo, dL_dw2_exo)</code></pre>

</details>

<details>
<summary>Afficher la solution 2 — Modifier une seule cible</summary>

<pre><code>import torch
w1_exo = torch.tensor(3., requires_grad=True)
w2_exo = torch.tensor(-1., requires_grad=True)
y_exo = w2_exo * (w1_exo * 2)
perte_exo = (y_exo - (-2.)).square()
perte_exo.backward()
assert (w1_exo.grad.item(), w2_exo.grad.item()) == (16.0, -48.0)
print(&#x27;gradients pour la cible -2 :&#x27;, w1_exo.grad.item(), w2_exo.grad.item())</code></pre>

</details>

